In [ ]:
# In[1]:

import pandas as pd
import pytz, datetime

# Load CSVs into DataFrames (stored for next steps)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_err = pd.read_csv("error_logs.csv")

# Ensure timestamp is integer type
for df in (df_metric, df_trace, df_log, df_err):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype('Int64')  # keep NA-safe integer type

# Compute incident window timestamps in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021, 3, 23, 16, 30, 0))
end_dt = tz.localize(datetime.datetime(2021, 3, 23, 17, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Helper to produce summary dict for a DataFrame
def summarize_df(df, kpi_col=None, sample_cols=None):
    total_row_count = int(df.shape[0])
    min_ts = int(df['timestamp'].min()) if total_row_count>0 else None
    max_ts = int(df['timestamp'].max()) if total_row_count>0 else None

    # top cmdb_id counts (up to 20)
    if 'cmdb_id' in df.columns:
        cmdb_counts = df['cmdb_id'].value_counts().head(20)
        unique_cmdb_id_counts = [(idx, int(cnt)) for idx, cnt in cmdb_counts.items()]
    else:
        unique_cmdb_id_counts = []

    # top KPI/Name counts (up to 20) if column provided and exists
    if kpi_col and kpi_col in df.columns:
        kpi_counts = df[kpi_col].value_counts().head(20)
        unique_kpi_name_counts = [(idx, int(cnt)) for idx, cnt in kpi_counts.items()]
    else:
        unique_kpi_name_counts = []

    # rows within incident window [start_ts, end_ts)
    if 'timestamp' in df.columns:
        mask = (df['timestamp'] >= start_ts) & (df['timestamp'] < end_ts)
        df_window = df.loc[mask].sort_values('timestamp')
        number_of_rows_within_incident_window = int(df_window.shape[0])
        # sample rows (up to 10) with selected columns if provided
        if sample_cols:
            sample_df = df_window.loc[:, [c for c in sample_cols if c in df_window.columns]].head(10)
        else:
            sample_df = df_window.head(10)
        # convert sample to list of records for compact display
        sample_rows_within_window = sample_df.to_dict('records')
    else:
        number_of_rows_within_incident_window = 0
        sample_rows_within_window = []

    return {
        "total_row_count": total_row_count,
        "min_timestamp": min_ts,
        "max_timestamp": max_ts,
        "unique_cmdb_id_counts": unique_cmdb_id_counts,
        "unique_kpi_name_counts": unique_kpi_name_counts,
        "number_of_rows_within_incident_window": number_of_rows_within_incident_window,
        "sample_rows_within_window": sample_rows_within_window
    }

# Summaries for each file with appropriate KPI column and sample columns
summary_metric = summarize_df(
    df_metric,
    kpi_col='kpi_name',
    sample_cols=['timestamp', 'cmdb_id', 'kpi_name', 'value']
)

summary_trace = summarize_df(
    df_trace,
    kpi_col='trace_name',
    sample_cols=['timestamp', 'cmdb_id', 'trace_name', 'value']
)

summary_log = summarize_df(
    df_log,
    kpi_col='log_name',
    sample_cols=['timestamp', 'cmdb_id', 'log_name', 'value']
)

# For error_logs, use 'message' as the "kpi" analogue and sample cols accordingly
summary_err = summarize_df(
    df_err,
    kpi_col='message',
    sample_cols=['timestamp', 'cmdb_id', 'message']
)

# Return the four summary dicts only (IPython will display them)
summary_metric, summary_trace, summary_log, summary_err

```
Out[1]:
```
# Summarization string (plain English) based on the previous execution results
summary = (
    "Summary of telemetry files (straightforward):\n"
    "- metric.csv: 22,479 rows (timestamps 1616488200 to 1616490000). "
    "Top CMDBs: Mysql02 (3866), Mysql01 (3737), Tomcat03 (2066), Tomcat02 (2024), Tomcat04 (2001). "
    "Top KPIs include app.mrt, app.cnt, app.sr, app.rr and various OSLinux metrics. "
    "22,006 rows fall inside the incident window (Mar 23, 2021 16:30–17:00 UTC+8). Sample rows show Tomcat03 OS/disk/memory metrics.\n\n"
    "- trace.csv: 7,266 rows (same timestamp range). "
    "Top CMDBs: MG02, MG01, Tomcat04/01/03/02 and IG nodes. "
    "Top trace names: trace.self.duration_mean, trace.self.duration_p95, trace.self.row_count (and many to/from MG/IG). "
    "7,056 rows are inside the incident window; sample rows show IG01 trace activity.\n\n"
    "- log.csv: 514 rows. "
    "Top CMDBs: Tomcat04, Tomcat03, Tomcat02, Tomcat01, apache01/02, Mysql01. "
    "Top log names: log.error_count and log.row_count. "
    "499 rows are inside the incident window; samples show mostly zero error_count and row_count values for Tomcat/Mysql.\n\n"
    "- error_logs.csv: 0 rows (no error log entries available).\n\n"
    "Overall conclusion: metric, trace, and log telemetry are largely present during the incident window (most rows fall within it), "
    "with prominent activity from Mysql and Tomcat nodes and trace activity around IG/MG components. No error_log records were captured."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

({'total_row_count': 22479, 'min_timestamp': 1616488200, 'max_timestamp': 1616490000, 'unique_cmdb_id_counts': [('Mysql02', 3866), ('Mysql01', 3737), ('Tomcat03', 2066), ('Tomcat02', 2024), ('Tomcat04', 2001), ('Tomcat01', 1966), ('MG01', 1585), ('MG02', 1576), ('Redis02', 1206), ('IG02', 337), ('IG01', 332), ('apache02', 143), ('apache01', 139), ('Redis01', 137), ('ServiceTest1', 124), ('ServiceTest10', 124), ('ServiceTest11', 124), ('ServiceTest7', 124), ('ServiceTest6', 124), ('ServiceTest5', 124)], 'unique_kpi_name_counts': [('app.mrt', 341), ('app.cnt', 341), ('app.sr', 341), ('app.rr', 341), ('OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 302), ('OSLinux-CPU_CPU_CPUCpuUtil', 281), ('OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 276), ('OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 216), ('OSLinux-CPU_CPU_CPUWio', 216), ('OSLinux-CPU_CPU_CPUidleutil', 216), ('OSLinux-CPU_CPU_CPULoad', 216), ('OSLinux-CPU_CPU_CPUUserTime', 216), ('OSLinux-CPU_CPU_CPUSysTime', 216), ('OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies', 215), ('OSLinux-OSLinux_ZABBIX_Host_Uptime', 215), ('OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount', 215), ('OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT', 211), ('OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 211), ('OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 152), ('OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 150)], 'number_of_rows_within_incident_window': 22006, 'sample_rows_within_window': [{'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'value': 7652.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps', 'value': 0.65}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite', 'value': 8.1333}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPercentBusy', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'kpi_name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps', 'value': 1.0}]}, {'total_row_count': 7266, 'min_timestamp': 1616488200, 'max_timestamp': 1616490000, 'unique_cmdb_id_counts': [('MG02', 1227), ('MG01', 1221), ('Tomcat04', 486), ('Tomcat01', 486), ('dockerB1', 486), ('dockerB2', 483), ('Tomcat03', 483), ('Tomcat02', 483), ('IG01', 480), ('IG02', 480), ('dockerA2', 477), ('dockerA1', 474)], 'unique_kpi_name_counts': [('trace.self.duration_mean', 372), ('trace.self.duration_p95', 372), ('trace.self.row_count', 372), ('trace.to_MG02.row_count', 247), ('trace.to_MG02.duration_mean', 247), ('trace.to_MG02.duration_p95', 247), ('trace.to_MG01.row_count', 245), ('trace.to_MG01.duration_mean', 245), ('trace.to_MG01.duration_p95', 245), ('trace.from_MG02.row_count', 124), ('trace.from_MG01.duration_mean', 124), ('trace.from_MG01.duration_p95', 124), ('trace.from_MG01.row_count', 124), ('trace.from_MG02.duration_p95', 124), ('trace.from_IG01.row_count', 124), ('trace.from_IG02.duration_mean', 124), ('trace.from_IG01.duration_mean', 124), ('trace.from_IG01.duration_p95', 124), ('trace.from_IG02.row_count', 124), ('trace.from_IG02.duration_p95', 124)], 'number_of_rows_within_incident_window': 7056, 'sample_rows_within_window': [{'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.self.row_count', 'value': 1505.0}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat03.duration_p95', 'value': 2.07225}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat03.duration_mean', 'value': 0.447091}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat02.row_count', 'value': 200.0}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat02.duration_p95', 'value': 2.6179}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat02.duration_mean', 'value': 0.622295}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat01.row_count', 'value': 183.0}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat01.duration_p95', 'value': 2.3753}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat01.duration_mean', 'value': 0.550809}, {'timestamp': 1616488200, 'cmdb_id': 'IG01', 'trace_name': 'trace.to_Tomcat03.row_count', 'value': 186.0}]}, {'total_row_count': 514, 'min_timestamp': 1616488200, 'max_timestamp': 1616490000, 'unique_cmdb_id_counts': [('Tomcat04', 85), ('Tomcat03', 84), ('Tomcat02', 83), ('Tomcat01', 82), ('apache01', 62), ('apache02', 62), ('Mysql01', 56)], 'unique_kpi_name_counts': [('log.error_count', 214), ('log.row_count', 214), ('log.gc_pause_max', 86)], 'number_of_rows_within_incident_window': 499, 'sample_rows_within_window': [{'timestamp': 1616488200, 'cmdb_id': 'Mysql01', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Mysql01', 'log_name': 'log.row_count', 'value': 27.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'value': 393.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat02', 'log_name': 'log.gc_pause_max', 'value': 0.05}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'value': 376.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'value': 0.06}, {'timestamp': 1616488200, 'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'value': 398.0}]}, {'total_row_count': 0, 'min_timestamp': None, 'max_timestamp': None, 'unique_cmdb_id_counts': [], 'unique_kpi_name_counts': [], 'number_of_rows_within_incident_window': 0, 'sample_rows_within_window': []})```
```

In [ ]:
# In[2]:

import pandas as pd, numpy as np, pytz, datetime

# 1) Restrict metric rows to possible root-cause components (keep full history for thresholds)
components = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
              'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
df_metric_restricted = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Ensure value is numeric and timestamp is integer
df_metric_restricted['value'] = pd.to_numeric(df_metric_restricted['value'], errors='coerce')
df_metric_restricted['timestamp'] = df_metric_restricted['timestamp'].astype('Int64')

# 2) Compute global P95 and P5 for each (cmdb_id, kpi_name) over full series
grp = df_metric_restricted.groupby(['cmdb_id', 'kpi_name'])['value']
df_thresholds = grp.agg(
    p95=lambda s: float(s.quantile(0.95)) if s.dropna().size>0 else np.nan,
    p5 = lambda s: float(s.quantile(0.05)) if s.dropna().size>0 else np.nan,
    total_points = lambda s: int(s.dropna().shape[0])
).reset_index()

# 3) Compute incident window timestamps in UTC+8 and filter restricted metrics to that window
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021,3,23,16,30,0))
end_dt   = tz.localize(datetime.datetime(2021,3,23,17,0,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_metric_incident = df_metric_restricted[(df_metric_restricted['timestamp'] >= start_ts) & 
                                         (df_metric_restricted['timestamp'] < end_ts)].copy()

# 4) For each (cmdb_id, kpi_name) in incident window compute anomaly counts and earliest timestamps
# Merge thresholds so we can compare
df_inc_with_thresh = df_metric_incident.merge(df_thresholds, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_th'))

def earliest_where(series_ts, series_cond):
    # series_ts and series_cond are aligned; return int timestamp or None
    if series_cond.any():
        return int(series_ts[series_cond].min())
    return None

agg_rows = []
grouped = df_inc_with_thresh.groupby(['cmdb_id','kpi_name'], sort=False)
for (cmdb, kpi), g in grouped:
    total_points_in_window = int(g['value'].dropna().shape[0])
    p95 = float(g['p95'].iloc[0]) if not pd.isna(g['p95'].iloc[0]) else np.nan
    p5  = float(g['p5'].iloc[0])  if not pd.isna(g['p5'].iloc[0]) else np.nan

    above_mask = g['value'] > p95 if not np.isnan(p95) else pd.Series([False]*g.shape[0], index=g.index)
    below_mask = g['value'] < p5  if not np.isnan(p5) else pd.Series([False]*g.shape[0], index=g.index)

    count_above_p95 = int(above_mask.sum())
    count_below_p5 = int(below_mask.sum())

    earliest_timestamp_above_p95 = earliest_where(g['timestamp'], above_mask)
    earliest_timestamp_below_p5 = earliest_where(g['timestamp'], below_mask)

    agg_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95': p95,
        'p5': p5,
        'total_points_in_window': total_points_in_window,
        'count_above_p95': count_above_p95,
        'count_below_p5': count_below_p5,
        'earliest_timestamp_above_p95': earliest_timestamp_above_p95,
        'earliest_timestamp_below_p5': earliest_timestamp_below_p5
    })

df_anomaly_counts = pd.DataFrame(agg_rows)

# Add severity key for ordering, then sort and limit to top 20
if not df_anomaly_counts.empty:
    df_anomaly_counts['severity'] = df_anomaly_counts[['count_above_p95','count_below_p5']].max(axis=1)
    df_anomaly_counts = df_anomaly_counts.sort_values(['severity','total_points_in_window'], ascending=[False, False]).drop(columns=['severity'])
else:
    df_anomaly_counts = df_anomaly_counts

# top 20 rows
df_anomaly_top20 = df_anomaly_counts.head(20).reset_index(drop=True)

# 5) Prepare return dict: list of dict rows (up to 20) plus total evaluated pairs count
evaluated_pairs_count = int(df_anomaly_counts.shape[0])
top_anomalies = df_anomaly_top20.to_dict(orient='records')

result = {'evaluated_pairs_count': evaluated_pairs_count, 'top_anomalies': top_anomalies}
result

```
Out[2]:
```
summary = (
    "Summary:\n"
    "- Evaluated 1,110 distinct (cmdb_id, kpi_name) pairs for the specified components.\n"
    "- Top anomalies (sampled up to 20) concentrate on IG, MG and Mysql nodes. Examples: IG01/IG02 JVM heap memory metrics, "
    "MG01/MG02 CPU metrics (load, sys/user/WIO/idle), and Mysql02 MySQL metrics (bytes sent, com_select, etc.).\n"
    "- Each top entry had 30 points in the incident window; the typical anomaly pattern in these top rows was small but present "
    "(commonly 2 points above the global P95 and 2 points below the global P5 within the window).\n"
    "- Earliest anomalous timestamps for these top entries fall inside the incident window (Mar 23, 2021 16:30–17:00 UTC+8).\n\n"
    "Conclusion: anomalies are concentrated on IG, MG and Mysql components and are primarily related to JVM heap and CPU-related KPIs. "
    "Counts per KPI in the window are small but recurring; these components merit further investigation."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'evaluated_pairs_count': 1110, 'top_anomalies': [{'cmdb_id': 'IG01', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'p95': 867836472.0, 'p5': 105522316.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488680.0, 'earliest_timestamp_below_p5': 1616488500.0}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'p95': 855134244.0, 'p5': 58346916.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488560.0, 'earliest_timestamp_below_p5': 1616489340.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'p95': 1001999768.0, 'p5': 241958424.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488200.0, 'earliest_timestamp_below_p5': 1616488800.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-CPU_CPU_CPULoad', 'p95': 1.46, 'p5': 1.0350000000000001, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488320.0, 'earliest_timestamp_below_p5': 1616488800.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-CPU_CPU_CPUSysTime', 'p95': 1.12745, 'p5': 0.8278369999999999, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488620.0, 'earliest_timestamp_below_p5': 1616488440.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-CPU_CPU_CPUUserTime', 'p95': 26.41795, 'p5': 26.074272999999998, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488200.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-CPU_CPU_CPUWio', 'p95': 0.802834, 'p5': 0.039536, 'total_points_in_window': 30, 'count_above_p95': 1, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489280.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-CPU_CPU_CPUidleutil', 'p95': 72.3823, 'p5': 71.22325000000001, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616489160.0, 'earliest_timestamp_below_p5': 1616489280.0}, {'cmdb_id': 'MG01', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'p95': 7689594.0, 'p5': 7687968.5, 'total_points_in_window': 30, 'count_above_p95': 1, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489940.0, 'earliest_timestamp_below_p5': 1616488200.0}, {'cmdb_id': 'MG02', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'p95': 954605016.0, 'p5': 179014164.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488440.0, 'earliest_timestamp_below_p5': 1616488800.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPULoad', 'p95': 1.395, 'p5': 1.025, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489700.0, 'earliest_timestamp_below_p5': 1616488800.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPUSysTime', 'p95': 1.18615, 'p5': 0.8216245, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488200.0, 'earliest_timestamp_below_p5': 1616489040.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPUUserTime', 'p95': 26.3906, 'p5': 26.059649, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616488200.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPUWio', 'p95': 0.44513400000000003, 'p5': 0.0458, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616488980.0, 'earliest_timestamp_below_p5': 1616488500.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-CPU_CPU_CPUidleutil', 'p95': 72.36125, 'p5': 71.4205475, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489160.0, 'earliest_timestamp_below_p5': 1616488200.0}, {'cmdb_id': 'MG02', 'kpi_name': 'OSLinux-OSLinux_ZABBIX_Host_Uptime', 'p95': 7689010.0, 'p5': 7687382.5, 'total_points_in_window': 30, 'count_above_p95': 1, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489940.0, 'earliest_timestamp_below_p5': 1616488200.0}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Binlog Cache Use', 'p95': 322037764.35, 'p5': 143.51449, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489820.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Bytes Sent', 'p95': 184143560390.05, 'p5': 85835.69326, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489820.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Com Insert', 'p95': 343931794.6, 'p5': 162.3951, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489820.0, 'earliest_timestamp_below_p5': 1616489160.0}, {'cmdb_id': 'Mysql02', 'kpi_name': 'Mysql-MySQL_3306_Com Select', 'p95': 544554040.65, 'p5': 252.03173500000003, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489820.0, 'earliest_timestamp_below_p5': 1616489160.0}]}```
```

In [ ]:
# In[3]:

import pandas as pd, numpy as np

# Reuse existing kernel variables: df_metric_incident, df_thresholds, df_anomaly_counts

# 1) Select top 20 pairs from df_anomaly_counts (already ordered previously)
top_pairs = df_anomaly_counts.head(20)[['cmdb_id', 'kpi_name']].drop_duplicates().reset_index(drop=True)

runs = []  # will collect all detected runs (including ones that may be filtered later)
pairs_with_any_run = set()

# Helper to get thresholds for a pair using df_thresholds (ensures using global thresholds)
thr_idx = df_thresholds.set_index(['cmdb_id','kpi_name'])

for _, row in top_pairs.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    # lookup thresholds; if missing, skip processing the series (no thresholds -> no runs)
    try:
        p95 = float(thr_idx.loc[(cmdb, kpi), 'p95'])
        p5  = float(thr_idx.loc[(cmdb, kpi), 'p5'])
    except Exception:
        # missing thresholds, note zero runs (no further action)
        continue

    # 1) extract incident-window ordered series
    ser = df_metric_incident[(df_metric_incident['cmdb_id'] == cmdb) & 
                             (df_metric_incident['kpi_name'] == kpi)][['timestamp','value']].sort_values('timestamp').reset_index(drop=True)

    if ser.empty:
        continue

    # 2) mark each point
    def mark_val(v):
        if pd.isna(v):
            return 'normal'
        if not pd.isna(p95) and v > p95:
            return 'above'
        if not pd.isna(p5) and v < p5:
            return 'below'
        return 'normal'
    ser['mark'] = ser['value'].apply(mark_val)

    # 3) identify consecutive runs of 'above' or 'below'
    # Create group ids for consecutive equal marks
    ser['grp'] = (ser['mark'] != ser['mark'].shift()).cumsum()
    grouped = ser.groupby('grp', sort=False)
    any_run_for_pair = False
    for _, g in grouped:
        m = g['mark'].iat[0]
        if m not in ('above','below'):
            continue
        any_run_for_pair = True
        start_ts = int(g['timestamp'].min())
        end_ts = int(g['timestamp'].max())
        run_length = int(g.shape[0])
        if m == 'above':
            extremal_value = float(g['value'].max())
            # compute breach_percent only if p95 is a valid positive number
            if (not pd.isna(p95)) and (p95 != 0):
                breach_percent = (extremal_value - p95) / p95
            else:
                breach_percent = np.nan
        else:  # below
            extremal_value = float(g['value'].min())
            if (not pd.isna(p5)) and (p5 != 0):
                breach_percent = (p5 - extremal_value) / p5
            else:
                breach_percent = np.nan

        runs.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'breach_type': m,
            'start_timestamp': start_ts,
            'end_timestamp': end_ts,
            'run_length': run_length,
            'extremal_value': extremal_value,
            'p95': p95,
            'p5': p5,
            'breach_percent': float(breach_percent) if not pd.isna(breach_percent) else np.nan
        })
    if any_run_for_pair:
        pairs_with_any_run.add((cmdb,kpi))

# 4) Build full detected runs DataFrame and filter candidate faults
df_detected_runs = pd.DataFrame(runs)

# Apply filter: run_length >=2 AND breach_percent > 0.5
if not df_detected_runs.empty:
    df_detected_runs['breach_percent'] = pd.to_numeric(df_detected_runs['breach_percent'], errors='coerce')
    df_candidate_faults = df_detected_runs[(df_detected_runs['run_length'] >= 2) & (df_detected_runs['breach_percent'] > 0.5)].copy()
else:
    df_candidate_faults = pd.DataFrame(columns=df_detected_runs.columns)

# 5) Prepare compact result
total_pairs_examined = int(top_pairs.shape[0])
total_pairs_with_faults = int(df_candidate_faults[['cmdb_id','kpi_name']].drop_duplicates().shape[0])

# Order candidate faults by breach_percent desc and limit to top 20
df_candidate_faults = df_candidate_faults.sort_values('breach_percent', ascending=False).head(20).reset_index(drop=True)

# Round breach_percent reasonably (4 decimals) and run_length keep integer
if not df_candidate_faults.empty:
    df_candidate_faults['breach_percent'] = df_candidate_faults['breach_percent'].round(4)

# Select & order fields for output
output_cols = ['cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp',
               'run_length','extremal_value','p95','p5','breach_percent']
fault_runs = df_candidate_faults.loc[:, output_cols]

result = {
    'total_pairs_examined': total_pairs_examined,
    'total_pairs_with_faults': total_pairs_with_faults,
    'fault_runs': fault_runs.to_dict(orient='records')
}

# Return compact result; df_detected_runs is stored in kernel for later steps
result

```
Out[3]:
```
summary = (
    "Summary:\n"
    "- Examined the top 20 (cmdb_id, kpi_name) pairs from the incident window.\n"
    "- No fault runs met the filter criteria (run_length >= 2 AND breach_percent > 0.5).\n"
    "- In plain terms: none of these top 20 series showed a sustained (>=2 consecutive points) extreme breach exceeding 50% relative to their global P95/P5 thresholds.\n"
    "- Note: df_detected_runs (stored in the kernel) contains all detected runs before filtering if you want to inspect borderline cases or adjust thresholds."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

{'total_pairs_examined': 20, 'total_pairs_with_faults': 0, 'fault_runs': []}```
```

In [ ]:
# In[4]:

import pandas as pd, numpy as np

# Ensure df_detected_runs exists; if not, create empty DataFrame with expected columns
expected_cols = ['cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp',
                 'run_length','extremal_value','p95','p5','breach_percent']
if 'df_detected_runs' not in globals():
    df_detected_runs = pd.DataFrame(columns=expected_cols)

# Normalize types
if not df_detected_runs.empty:
    df_detected_runs['run_length'] = pd.to_numeric(df_detected_runs['run_length'], errors='coerce').fillna(0).astype(int)
    df_detected_runs['breach_percent'] = pd.to_numeric(df_detected_runs['breach_percent'], errors='coerce')
    df_detected_runs['extremal_value'] = pd.to_numeric(df_detected_runs['extremal_value'], errors='coerce')
    df_detected_runs['p95'] = pd.to_numeric(df_detected_runs['p95'], errors='coerce')
    df_detected_runs['p5'] = pd.to_numeric(df_detected_runs['p5'], errors='coerce')

#  Filter with permissive criteria: run_length >=2 OR breach_percent >= 0.2
if df_detected_runs.empty:
    df_candidate_faults = df_detected_runs.copy()
else:
    mask = (df_detected_runs['run_length'] >= 2) | (df_detected_runs['breach_percent'] >= 0.2)
    df_candidate_faults = df_detected_runs.loc[mask].copy()

# Prepare compact top-20 list ordered by breach_percent desc
if not df_candidate_faults.empty:
    df_candidate_faults['breach_percent'] = pd.to_numeric(df_candidate_faults['breach_percent'], errors='coerce')
    df_candidate_faults_top20 = (df_candidate_faults
                                .sort_values(['breach_percent','run_length'], ascending=[False, False])
                                .loc[:, expected_cols]
                                .head(20)
                                .reset_index(drop=True))
    # Round breach_percent reasonably
    df_candidate_faults_top20['breach_percent'] = df_candidate_faults_top20['breach_percent'].round(4)
else:
    df_candidate_faults_top20 = df_candidate_faults.copy()

# Aggregate counts per cmdb_id (top 10)
if not df_candidate_faults.empty:
    counts_top10 = (df_candidate_faults
                    .groupby('cmdb_id')
                    .size()
                    .reset_index(name='candidate_run_count')
                    .sort_values('candidate_run_count', ascending=False)
                    .head(10)
                    .reset_index(drop=True))
else:
    counts_top10 = pd.DataFrame(columns=['cmdb_id','candidate_run_count'])

# Store full filtered candidates in kernel variable for next steps (rule requirement)
# df_candidate_faults already holds the filtered candidates

# Display compact outputs
df_candidate_faults_top20, counts_top10

```
Out[4]:
```
Summary of candidate fault runs (more permissive criteria: run_length >=2 OR breach_percent >=0.2):

- Total candidate runs shown (top 20 sample): 5 distinct runs listed in the compact top results.
- Per-component counts (top 10): MG01: 3 runs, MG02: 2 runs, IG01: 1 run.

Top candidates (key points):
- IG01 — JVM-Memory_7778_JVM_Memory_HeapMemoryUsed: a single-point "below" breach with breach_percent ≈ 0.324 (≈32.4% below the global P5). This is a high-relative breach but only one consecutive point.
- MG02 — OSLinux-CPU_CPU_CPULoad: a 2-point "above" run, breach_percent ≈ 0.226 (≈22.6% above P95). This is a short but sustained CPU-high run.
- MG01 — OSLinux-CPU_CPU_CPUWio: a 2-point "below" run, breach_percent ≈ 0.158 (≈15.8% below P5). Run length is 2 but relative breach is modest.
- MG01 — OSLinux-CPU_CPU_CPULoad: a 2-point "below" run, breach_percent ≈ 0.024 (very small relative breach).
- MG02 / MG01 — OSLinux-OSLinux_ZABBIX_Host_Uptime: 2-point "below" runs with essentially zero relative breach (breach_percent ≈ 0).

Interpretation / recommendation:
- The most notable signals are IG01 JVM memory (large relative drop but single point) and MG02 CPU load (2 consecutive points significantly above P95). These are the highest-priority candidates for investigation.
- Several MG01/MG02 CPU-related runs exist but with smaller relative breaches or marginal severity; they may indicate minor fluctuations or noise.
- Next steps: inspect the raw series and related traces/logs around the timestamps for IG01 JVM and MG02 CPU load to determine whether these represent real faults or measurement artifacts.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                   kpi_name breach_type  start_timestamp  end_timestamp  run_length  extremal_value           p95            p5  breach_percent
0    IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed       below       1616488500     1616488500           1    7.133570e+07  8.678365e+08  1.055223e+08          0.3240
1    MG02                    OSLinux-CPU_CPU_CPULoad       above       1616489700     1616489760           2    1.710000e+00  1.395000e+00  1.025000e+00          0.2258
2    MG01                     OSLinux-CPU_CPU_CPUWio       below       1616489160     1616489220           2    3.328200e-02  8.028340e-01  3.953600e-02          0.1582
3    MG01                    OSLinux-CPU_CPU_CPULoad       below       1616488800     1616488860           2    1.010000e+00  1.460000e+00  1.035000e+00          0.0242
4    MG02         OSLinux-OSLinux_ZABBIX_Host_Uptime       below       1616488200     1616488260           2    7.687292e+06  7.689010e+06  7.687382e+06          0.0000
5    MG01         OSLinux-OSLinux_ZABBIX_Host_Uptime       below       1616488200     1616488260           2    7.687879e+06  7.689594e+06  7.687968e+06          0.0000,   cmdb_id  candidate_run_count
0    MG01                    3
1    MG02                    2
2    IG01                    1)```
```

In [ ]:
# In[5]:

import pandas as pd, numpy as np, pytz, datetime

# 1) Restrict df_trace to specified components and compute global P95/P5 per (cmdb_id, trace_name)
components = ['MG01','MG02','IG01']
df_trace_restricted = df_trace[df_trace['cmdb_id'].isin(components)].copy()

# normalize types
df_trace_restricted['value'] = pd.to_numeric(df_trace_restricted['value'], errors='coerce')
df_trace_restricted['timestamp'] = df_trace_restricted['timestamp'].astype('Int64')

grp = df_trace_restricted.groupby(['cmdb_id','trace_name'])['value']
df_trace_thresholds = grp.agg(
    p95 = lambda s: float(s.quantile(0.95)) if s.dropna().size>0 else np.nan,
    p5  = lambda s: float(s.quantile(0.05)) if s.dropna().size>0 else np.nan,
    total_points = lambda s: int(s.dropna().shape[0])
).reset_index()

# 2) Compute incident window and filter trace rows to that window
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021,3,23,16,30,0))
end_dt   = tz.localize(datetime.datetime(2021,3,23,17,0,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_trace_incident = df_trace_restricted[(df_trace_restricted['timestamp'] >= start_ts) &
                                       (df_trace_restricted['timestamp'] < end_ts)].copy()

# Merge thresholds into incident rows so we can compare
df_trace_inc_with_thresh = df_trace_incident.merge(df_trace_thresholds, on=['cmdb_id','trace_name'], how='left')

# Helper for earliest timestamp where condition holds
def earliest_where(series_ts, series_cond):
    if series_cond.any():
        return int(series_ts[series_cond].min())
    return None

# 2 continued) Aggregate per (cmdb_id, trace_name) within incident window
agg_rows = []
grouped = df_trace_inc_with_thresh.groupby(['cmdb_id','trace_name'], sort=False)
for (cmdb, trace_name), g in grouped:
    total_points_in_window = int(g['value'].dropna().shape[0])
    p95 = float(g['p95'].iloc[0]) if not pd.isna(g['p95'].iloc[0]) else np.nan
    p5  = float(g['p5'].iloc[0])  if not pd.isna(g['p5'].iloc[0]) else np.nan

    above_mask = g['value'] > p95 if not np.isnan(p95) else pd.Series([False]*g.shape[0], index=g.index)
    below_mask = g['value'] < p5  if not np.isnan(p5) else pd.Series([False]*g.shape[0], index=g.index)

    count_above_p95 = int(above_mask.sum())
    count_below_p5 = int(below_mask.sum())
    earliest_timestamp_above_p95 = earliest_where(g['timestamp'], above_mask)
    earliest_timestamp_below_p5 = earliest_where(g['timestamp'], below_mask)

    agg_rows.append({
        'cmdb_id': cmdb,
        'trace_name': trace_name,
        'p95': p95,
        'p5': p5,
        'total_points_in_window': total_points_in_window,
        'count_above_p95': count_above_p95,
        'count_below_p5': count_below_p5,
        'earliest_timestamp_above_p95': earliest_timestamp_above_p95,
        'earliest_timestamp_below_p5': earliest_timestamp_below_p5
    })

df_trace_summary = pd.DataFrame(agg_rows)

# 3) Return compact result limited to top 20 rows ordered by max(count_above_p95, count_below_p5) desc
if not df_trace_summary.empty:
    df_trace_summary['severity'] = df_trace_summary[['count_above_p95','count_below_p5']].max(axis=1)
    df_trace_summary = df_trace_summary.sort_values(['severity','total_points_in_window'], ascending=[False, False]).drop(columns=['severity'])
else:
    df_trace_summary = df_trace_summary

evaluated_pairs_count = int(df_trace_summary.shape[0])
df_trace_summary_top20 = df_trace_summary.head(20).reset_index(drop=True)

# Keep df_trace_thresholds and df_trace_incident in kernel for follow-ups (already named)
# Output compact results
df_trace_summary_top20, evaluated_pairs_count

```
Out[5]:
```
Summary (plain English):

- I evaluated 102 (cmdb_id, trace_name) pairs for MG01, MG02, and IG01 during the incident window.
- The top-20 results are dominated by IG01 trace metrics (15 of the top 20) and several MG01 traces (5 of the top 20).
- Each top row had 30 data points in the incident window. Many trace series show small but measurable anomalies: typically 2 points above the global P95 and 1–2 points below the global P5 within the window.
- Affected trace types include duration_mean, duration_p95, and row_count, especially for IG01’s self traces and IG01→TomcatX traces (Tomcat01–04). MG01 shows anomalies on traces from Tomcat01/02.
- Earliest above-P95 timestamps for these top entries are inside the incident window (timestamps like 1616488320, 1616488560, 1616488740, 1616489820, etc.).
- Interpretation and next steps: IG01 is the most active source of trace-level anomalies (latency and counts) toward the Tomcat tier, with MG01 also showing anomalous inbound traces. I recommend investigating IG01→Tomcat interactions and MG01 inbound traces at the listed timestamps, and correlating these with the metric-level signals (e.g., MG CPU and IG JVM memory) and logs to determine root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                         trace_name          p95           p5  total_points_in_window  count_above_p95  count_below_p5  earliest_timestamp_above_p95  earliest_timestamp_below_p5
0     IG01           trace.self.duration_mean     0.622221     0.257422                      30                2               2                    1616489820                 1.616488e+09
1     IG01            trace.self.duration_p95     2.270150     1.064800                      30                2               2                    1616488740                 1.616488e+09
2     IG01               trace.self.row_count  1939.000000  1510.500000                      30                2               1                    1616488560                 1.616488e+09
3     IG01    trace.to_Tomcat01.duration_mean     0.572414     0.237946                      30                2               1                    1616488740                 1.616488e+09
4     IG01     trace.to_Tomcat01.duration_p95     3.071825     1.057800                      30                2               1                    1616488740                 1.616488e+09
5     IG01        trace.to_Tomcat01.row_count   249.000000   179.000000                      30                2               1                    1616489340                 1.616488e+09
6     IG01    trace.to_Tomcat02.duration_mean     0.738036     0.256273                      30                1               2                    1616489880                 1.616488e+09
7     IG01     trace.to_Tomcat02.duration_p95     3.153575     1.062375                      30                1               2                    1616489880                 1.616488e+09
8     IG01        trace.to_Tomcat02.row_count   250.500000   186.500000                      30                2               1                    1616488560                 1.616489e+09
9     IG01    trace.to_Tomcat03.duration_mean     0.533056     0.244261                      30                1               2                    1616489880                 1.616488e+09
10    IG01     trace.to_Tomcat03.duration_p95     2.235225     1.055650                      30                2               2                    1616488320                 1.616488e+09
11    IG01        trace.to_Tomcat03.row_count   257.000000   191.500000                      30                2               1                    1616488440                 1.616488e+09
12    IG01    trace.to_Tomcat04.duration_mean     0.577558     0.242592                      30                2               2                    1616488320                 1.616488e+09
13    IG01     trace.to_Tomcat04.duration_p95     2.897450     1.036600                      30                2               1                    1616488320                 1.616488e+09
14    IG01        trace.to_Tomcat04.row_count   246.000000   179.500000                      30                2               1                    1616488560                 1.616488e+09
15    MG01  trace.from_Tomcat01.duration_mean     0.332070     0.110473                      30                2               1                    1616488740                 1.616489e+09
16    MG01   trace.from_Tomcat01.duration_p95     2.088325     1.020675                      30                2               1                    1616488740                 1.616488e+09
17    MG01      trace.from_Tomcat01.row_count   253.000000   177.500000                      30                2               1                    1616488440                 1.616488e+09
18    MG01  trace.from_Tomcat02.duration_mean     0.365606     0.083974                      30                2               1                    1616488740                 1.616488e+09
19    MG01   trace.from_Tomcat02.duration_p95     2.089350     0.581200                      30                2               1                    1616488320                 1.616488e+09, 102)```
```

In [ ]:
# In[6]:

import pandas as pd, numpy as np, pytz, datetime

# 1) Restrict df_log to specified service-tier components and compute global P95/P5 per (cmdb_id, log_name)
service_components = ['Tomcat01','Tomcat02','Tomcat03','Tomcat04','Mysql01','Mysql02']
df_log_restricted = df_log[df_log['cmdb_id'].isin(service_components)].copy()

# Normalize types
df_log_restricted['value'] = pd.to_numeric(df_log_restricted['value'], errors='coerce')
df_log_restricted['timestamp'] = df_log_restricted['timestamp'].astype('Int64')

grp = df_log_restricted.groupby(['cmdb_id','log_name'])['value']
df_log_thresholds = grp.agg(
    p95 = lambda s: float(s.quantile(0.95)) if s.dropna().size>0 else np.nan,
    p5  = lambda s: float(s.quantile(0.05)) if s.dropna().size>0 else np.nan,
    total_points = lambda s: int(s.dropna().shape[0])
).reset_index()

# 2) Compute incident window and filter df_log to that window
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2021,3,23,16,30,0))
end_dt   = tz.localize(datetime.datetime(2021,3,23,17,0,0))
start_ts = int(start_dt.timestamp())
end_ts   = int(end_dt.timestamp())

df_log_incident = df_log_restricted[(df_log_restricted['timestamp'] >= start_ts) & 
                                   (df_log_restricted['timestamp'] < end_ts)].copy()

# 3) For each (cmdb_id, log_name) in df_log_incident compute required stats
df_log_inc_with_thresh = df_log_incident.merge(df_log_thresholds, on=['cmdb_id','log_name'], how='left')

def earliest_where(series_ts, series_cond):
    if series_cond.any():
        return int(series_ts[series_cond].min())
    return None

agg_rows = []
grouped = df_log_inc_with_thresh.groupby(['cmdb_id','log_name'], sort=False)
for (cmdb, log_name), g in grouped:
    total_points_in_window = int(g['value'].dropna().shape[0])
    p95 = float(g['p95'].iloc[0]) if not pd.isna(g['p95'].iloc[0]) else np.nan
    p5  = float(g['p5'].iloc[0])  if not pd.isna(g['p5'].iloc[0]) else np.nan

    above_mask = g['value'] > p95 if not np.isnan(p95) else pd.Series([False]*g.shape[0], index=g.index)
    below_mask = g['value'] < p5  if not np.isnan(p5) else pd.Series([False]*g.shape[0], index=g.index)

    count_above_p95 = int(above_mask.sum())
    count_below_p5 = int(below_mask.sum())
    earliest_timestamp_above_p95 = earliest_where(g['timestamp'], above_mask)
    earliest_timestamp_below_p5 = earliest_where(g['timestamp'], below_mask)

    agg_rows.append({
        'cmdb_id': cmdb,
        'log_name': log_name,
        'p95': p95,
        'p5': p5,
        'total_points_in_window': total_points_in_window,
        'count_above_p95': count_above_p95,
        'count_below_p5': count_below_p5,
        'earliest_timestamp_above_p95': earliest_timestamp_above_p95,
        'earliest_timestamp_below_p5': earliest_timestamp_below_p5
    })

df_log_anomalies = pd.DataFrame(agg_rows)

# 4) Extract raw df_log_incident rows where log_name == 'log.error_count' and value > 0
df_log_errors_nonzero = df_log_incident[(df_log_incident['log_name'] == 'log.error_count') & (df_log_incident['value'] > 0)].sort_values('timestamp').head(20).reset_index(drop=True)

# 5) Prepare compact payload: evaluated_pairs_count and top_log_anomalies (up to 20 rows ordered by severity desc)
if not df_log_anomalies.empty:
    df_log_anomalies['severity'] = df_log_anomalies[['count_above_p95','count_below_p5']].max(axis=1)
    df_log_anomalies = df_log_anomalies.sort_values(['severity','total_points_in_window'], ascending=[False, False]).drop(columns=['severity'])
else:
    df_log_anomalies = df_log_anomalies

evaluated_pairs_count = int(df_log_anomalies.shape[0])
df_log_anomalies_top20 = df_log_anomalies.head(20).reset_index(drop=True)

# Round numeric thresholds for compact display
if not df_log_anomalies_top20.empty:
    df_log_anomalies_top20['p95'] = pd.to_numeric(df_log_anomalies_top20['p95'], errors='coerce')
    df_log_anomalies_top20['p5']  = pd.to_numeric(df_log_anomalies_top20['p5'], errors='coerce')

# Store required kernel variables for follow-up: df_log_thresholds, df_log_incident, df_log_anomalies (already set)

# Final compact payload
result = {
    'evaluated_pairs_count': evaluated_pairs_count,
    'top_log_anomalies': df_log_anomalies_top20.loc[:, ['cmdb_id','log_name','p95','p5','total_points_in_window','count_above_p95','count_below_p5','earliest_timestamp_above_p95','earliest_timestamp_below_p5']].to_dict(orient='records'),
    'raw_nonzero_error_count_rows': df_log_errors_nonzero.loc[:, ['timestamp','cmdb_id','log_name','value']].to_dict(orient='records')
}

result, df_log_thresholds.head(20), df_log_incident.shape, df_log_anomalies.shape

```
Out[6]:
```
Summary (plain English):

- I evaluated 14 (cmdb_id, log_name) pairs for Tomcat01–04 and Mysql01/02 during the incident window.
- Dataset sizes saved: df_log_incident contains 379 log rows in the incident window; df_log_thresholds and df_log_anomalies are stored for follow-up.
- Top log anomalies (up to 20 shown; 14 pairs evaluated):
  - Tomcat01/02/03/04: log.row_count shows small anomalies — typically 30 points in-window, with 2 points above the global P95 and 1 point below P5 for each Tomcat. Earliest above-P95 timestamps are between 1616488560 and 1616489280 (inside the incident window).
  - Tomcat04, Tomcat02, Tomcat03, Tomcat01: log.gc_pause_max had a few above-P95 points (1–2) — potential brief GC pauses (p95 values ~0.08–0.117).
  - Mysql01: log.row_count had 1 above-P95 point in the window (total_points_in_window = 27).
- log.error_count: global p95/p5 are 0 for Tomcat and Mysql in this dataset, and no raw non-zero error_count rows were found in the incident window (raw_nonzero_error_count_rows is empty).
- Interpretation / next steps:
  - The most prominent log-level signals are increased row_count spikes on all Tomcat instances (2 above-P95 points each) and a few GC pause spikes. These align with observed trace and metric signals on the Tomcat/MG/IG tiers and warrant investigating request/throughput spikes and GC around the noted timestamps.
  - No explicit application error log entries were captured during the window (no non-zero error_count rows), so focus on performance/latency and throughput diagnostics rather than explicit error messages.

The original code execution output of IPython Kernel is also provided below for reference:

({'evaluated_pairs_count': 14, 'top_log_anomalies': [{'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'p95': 484.5, 'p5': 383.5, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616488740.0, 'earliest_timestamp_below_p5': 1616488320.0}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'p95': 503.0, 'p5': 381.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616488560.0, 'earliest_timestamp_below_p5': 1616488200.0}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'p95': 502.5, 'p5': 389.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616488560.0, 'earliest_timestamp_below_p5': 1616489580.0}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.row_count', 'p95': 471.0, 'p5': 392.0, 'total_points_in_window': 30, 'count_above_p95': 2, 'count_below_p5': 1, 'earliest_timestamp_above_p95': 1616489280.0, 'earliest_timestamp_below_p5': 1616488320.0}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.gc_pause_max', 'p95': 0.11699999999999995, 'p5': 0.021, 'total_points_in_window': 23, 'count_above_p95': 2, 'count_below_p5': 2, 'earliest_timestamp_above_p95': 1616489160.0, 'earliest_timestamp_below_p5': 1616488860.0}, {'cmdb_id': 'Mysql01', 'log_name': 'log.row_count', 'p95': 12.0, 'p5': 1.0, 'total_points_in_window': 27, 'count_above_p95': 1, 'count_below_p5': 0, 'earliest_timestamp_above_p95': 1616488200.0, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.gc_pause_max', 'p95': 0.09, 'p5': 0.02, 'total_points_in_window': 21, 'count_above_p95': 1, 'count_below_p5': 0, 'earliest_timestamp_above_p95': 1616489280.0, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'p95': 0.08, 'p5': 0.02, 'total_points_in_window': 21, 'count_above_p95': 1, 'count_below_p5': 0, 'earliest_timestamp_above_p95': 1616489280.0, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.gc_pause_max', 'p95': 0.09050000000000001, 'p5': 0.02, 'total_points_in_window': 20, 'count_above_p95': 1, 'count_below_p5': 0, 'earliest_timestamp_above_p95': 1616488440.0, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'p95': 0.0, 'p5': 0.0, 'total_points_in_window': 30, 'count_above_p95': 0, 'count_below_p5': 0, 'earliest_timestamp_above_p95': nan, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'p95': 0.0, 'p5': 0.0, 'total_points_in_window': 30, 'count_above_p95': 0, 'count_below_p5': 0, 'earliest_timestamp_above_p95': nan, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'p95': 0.0, 'p5': 0.0, 'total_points_in_window': 30, 'count_above_p95': 0, 'count_below_p5': 0, 'earliest_timestamp_above_p95': nan, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.error_count', 'p95': 0.0, 'p5': 0.0, 'total_points_in_window': 30, 'count_above_p95': 0, 'count_below_p5': 0, 'earliest_timestamp_above_p95': nan, 'earliest_timestamp_below_p5': nan}, {'cmdb_id': 'Mysql01', 'log_name': 'log.error_count', 'p95': 0.0, 'p5': 0.0, 'total_points_in_window': 27, 'count_above_p95': 0, 'count_below_p5': 0, 'earliest_timestamp_above_p95': nan, 'earliest_timestamp_below_p5': nan}], 'raw_nonzero_error_count_rows': []},      cmdb_id          log_name       p95       p5  total_points
0    Mysql01   log.error_count    0.0000    0.000            28
1    Mysql01     log.row_count   12.0000    1.000            28
2   Tomcat01   log.error_count    0.0000    0.000            31
3   Tomcat01  log.gc_pause_max    0.0905    0.020            20
4   Tomcat01     log.row_count  484.5000  383.500            31
5   Tomcat02   log.error_count    0.0000    0.000            31
6   Tomcat02  log.gc_pause_max    0.0900    0.020            21
7   Tomcat02     log.row_count  503.0000  381.000            31
8   Tomcat03   log.error_count    0.0000    0.000            31
9   Tomcat03  log.gc_pause_max    0.0800    0.020            22
10  Tomcat03     log.row_count  502.5000  389.000            31
11  Tomcat04   log.error_count    0.0000    0.000            31
12  Tomcat04  log.gc_pause_max    0.1170    0.021            23
13  Tomcat04     log.row_count  471.0000  392.000            31, (379, 4), (14, 9))```
```

In [ ]:
# In[7]:

import pandas as pd, numpy as np

# Reuse kernel variables: df_metric_incident, df_trace_incident, df_log_incident,
# df_thresholds, df_trace_thresholds, df_log_thresholds, df_candidate_faults, df_detected_runs

# Helper to ensure existence
for varname in ['df_metric_incident','df_trace_incident','df_log_incident',
                'df_thresholds','df_trace_thresholds','df_log_thresholds',
                'df_candidate_faults','df_detected_runs']:
    if varname not in globals():
        globals()[varname] = pd.DataFrame()

# 1) Trace events: mark each breached row
if not df_trace_incident.empty:
    # merge thresholds
    df_t = df_trace_incident.merge(df_trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    def trace_mark(r):
        v = r['value']
        if pd.isna(v): 
            return None
        if (not pd.isna(r['p95'])) and (v > r['p95']):
            return 'above'
        if (not pd.isna(r['p5'])) and (v < r['p5']):
            return 'below'
        return None
    df_t['breach_type'] = df_t.apply(trace_mark, axis=1)
    df_trace_events = df_t[df_t['breach_type'].notna()].loc[:, ['timestamp','cmdb_id','trace_name','breach_type']].copy()
    df_trace_events['source'] = 'trace'
    df_trace_events = df_trace_events.rename(columns={'trace_name':'name'})  # align field name
else:
    df_trace_events = pd.DataFrame(columns=['timestamp','cmdb_id','name','breach_type','source'])

# 2) Log events: similar
if not df_log_incident.empty:
    df_l = df_log_incident.merge(df_log_thresholds, on=['cmdb_id','log_name'], how='left')
    def log_mark(r):
        v = r['value']
        if pd.isna(v):
            return None
        if (not pd.isna(r['p95'])) and (v > r['p95']):
            return 'above'
        if (not pd.isna(r['p5'])) and (v < r['p5']):
            return 'below'
        return None
    df_l['breach_type'] = df_l.apply(log_mark, axis=1)
    df_log_events = df_l[df_l['breach_type'].notna()].loc[:, ['timestamp','cmdb_id','log_name','breach_type']].copy()
    df_log_events['source'] = 'log'
    df_log_events = df_log_events.rename(columns={'log_name':'name'})
else:
    df_log_events = pd.DataFrame(columns=['timestamp','cmdb_id','name','breach_type','source'])

# 3) Metric run events from df_detected_runs and df_candidate_faults
metric_runs = pd.DataFrame()
# prefer df_detected_runs (all runs) and include candidate set for completeness
if not df_detected_runs.empty:
    metric_runs = pd.concat([metric_runs, df_detected_runs], ignore_index=True, sort=False)
if 'df_candidate_faults' in globals() and isinstance(df_candidate_faults, pd.DataFrame) and not df_candidate_faults.empty:
    # ensure no duplicates: concat then drop duplicates by all key fields
    metric_runs = pd.concat([metric_runs, df_candidate_faults], ignore_index=True, sort=False)
# dedupe if metric_runs not empty
if not metric_runs.empty:
    metric_runs = metric_runs.drop_duplicates(subset=['cmdb_id','kpi_name','breach_type','start_timestamp','end_timestamp','run_length','extremal_value'], keep='first').reset_index(drop=True)

metric_event_rows = []
if not metric_runs.empty:
    for _, r in metric_runs.iterrows():
        cmdb = r.get('cmdb_id')
        name = r.get('kpi_name')
        btype = r.get('breach_type')
        start_ts = int(r.get('start_timestamp')) if pd.notna(r.get('start_timestamp')) else None
        end_ts = int(r.get('end_timestamp')) if pd.notna(r.get('end_timestamp')) else start_ts
        run_len = int(r.get('run_length')) if pd.notna(r.get('run_length')) else 1

        if start_ts is None:
            continue
        if run_len >= 2:
            # attempt to infer step; fallback to 60s
            if (end_ts is not None) and (end_ts > start_ts):
                step = int(round((end_ts - start_ts) / max(run_len - 1, 1)))
                step = max(step, 1)
            else:
                step = 60
            # create timestamps sequence; ensure not huge
            ts_seq = list(range(start_ts, end_ts + 1, step))
            if len(ts_seq) == 0:
                ts_seq = [start_ts]
        else:
            ts_seq = [start_ts]
        for ts in ts_seq:
            metric_event_rows.append({'timestamp': int(ts), 'cmdb_id': cmdb, 'name': name, 'breach_type': btype, 'source': 'metric'})
metric_events = pd.DataFrame(metric_event_rows, columns=['timestamp','cmdb_id','name','breach_type','source'])

# 4) Aggregate all events
all_events = pd.concat([df_trace_events[['timestamp','cmdb_id','name','breach_type','source']],
                        df_log_events[['timestamp','cmdb_id','name','breach_type','source']],
                        metric_events[['timestamp','cmdb_id','name','breach_type','source']]],
                       ignore_index=True, sort=False)

# If no events, produce empty correlation
if all_events.empty:
    df_timestamp_correlation = pd.DataFrame(columns=['timestamp','distinct_component_count','total_event_count','events'])
    df_timestamp_correlation_top10 = df_timestamp_correlation.copy()
else:
    # build tuple representation for event list
    all_events['event_tuple'] = list(zip(all_events['cmdb_id'], all_events['source'], all_events['name'], all_events['breach_type']))

    # group by timestamp
    grp = all_events.groupby('timestamp')
    agg_rows = []
    for ts, g in grp:
        distinct_component_count = int(g['cmdb_id'].nunique())
        total_event_count = int(g.shape[0])
        events_list = list(g['event_tuple'].iloc[0:50])  # already limited
        agg_rows.append({'timestamp': int(ts),
                         'distinct_component_count': distinct_component_count,
                         'total_event_count': total_event_count,
                         'events': events_list})
    df_timestamp_correlation = pd.DataFrame(agg_rows)
    # sort by distinct_component_count desc then total_event_count desc
    df_timestamp_correlation = df_timestamp_correlation.sort_values(['distinct_component_count','total_event_count'], ascending=[False, False]).reset_index(drop=True)
    df_timestamp_correlation_top10 = df_timestamp_correlation.head(10).copy()

# Store in kernel variable for follow-up
# (already assigned to df_timestamp_correlation)

# Final compact display: top10 correlation and overall shape
df_timestamp_correlation_top10, df_timestamp_correlation.shape

```
Out[7]:
```
import pytz, datetime

# Build a plain-English summary based on df_timestamp_correlation (top 10 already computed)
top = df_timestamp_correlation.head(10).copy() if 'df_timestamp_correlation' in globals() else None
tz = pytz.timezone('Asia/Shanghai')

if top is None or top.empty:
    summary = "No correlated events were found in the incident window."
else:
    lines = []
    lines.append("Correlation summary (incident window Mar 23, 2021 16:30–17:00 UTC+8):")
    lines.append("")
    lines.append("Top correlated timestamps (UTC+8) with number of distinct components and total events:")
    for _, r in top.iterrows():
        ts = int(r['timestamp'])
        dt = datetime.datetime.fromtimestamp(ts, tz)
        timestr = dt.strftime("%Y-03-23 %H:%M:%S %Z")
        lines.append(f"- {timestr}: {r['distinct_component_count']} distinct components, {r['total_event_count']} events")
    lines.append("")
    lines.append("Key observations:")
    lines.append("- IG01 is the dominant source of trace-level anomalies across many top timestamps.")
    lines.append("- Significant correlation peaks occur around the listed timestamps (e.g., ~16:48–16:58 UTC+8), where multiple trace, metric, and log events co-occur.")
    lines.append("- MG nodes (MG01/MG02) also surface repeatedly, often with metric-level CPU and trace inbound anomalies that align with IG01 trace bursts.")
    lines.append("- Tomcat instances show correlated log.row_count spikes (throughput) at several timestamps that line up with the trace/metric signals.")
    lines.append("")
    lines.append("Actionable next steps:")
    lines.append("- Prioritize investigation of IG01 interactions with the Tomcat tier at the top timestamps (inspect traces and request patterns).")
    lines.append("- Correlate MG01/MG02 CPU spikes with IG01 trace bursts for potential downstream resource impact.")
    lines.append("- Check Tomcat request throughput and GC around these timestamps (already observed small GC spikes).")
    lines.append("")
    lines.append("Conclusion: multiple telemetry sources converge around a few clustered timestamps, with IG01 and MG nodes at the center; investigate those timestamps and components first.")
    summary = "\n".join(lines)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp  distinct_component_count  total_event_count                                             events
0  1616489280                         6                 23  [(IG01, trace, trace.to_Tomcat04.duration_mean...
1  1616489340                         6                 16  [(IG01, trace, trace.self.row_count, above), (...
2  1616488560                         6                 14  [(IG01, trace, trace.self.row_count, above), (...
3  1616488200                         5                 36  [(IG01, trace, trace.from_root.row_count, abov...
4  1616489160                         5                 23  [(MG01, trace, trace.from_Tomcat03.row_count, ...
5  1616488320                         5                 20  [(IG01, trace, trace.from_root.duration_mean, ...
6  1616489820                         5                 12  [(IG01, trace, trace.from_root.duration_mean, ...
7  1616489940                         5                 12  [(IG01, trace, trace.to_Tomcat03.row_count, ab...
8  1616488440                         4                 49  [(IG01, trace, trace.self.duration_mean, below...
9  1616488740                         4                 28  [(IG01, trace, trace.self.duration_p95, above)..., (30, 4))```
```

In [ ]:
# In[8]:

import pandas as pd, numpy as np
from collections import Counter

# Reuse df_timestamp_correlation, df_trace_incident, df_log_incident, df_metric_incident
# Ensure expected variables exist
for name in ['df_timestamp_correlation','df_trace_incident','df_log_incident','df_metric_incident']:
    if name not in globals():
        globals()[name] = pd.DataFrame()

# Take top 2 timestamps (already ordered)
top_ts_df = df_timestamp_correlation.head(2).reset_index(drop=True)
ts_list = top_ts_df['timestamp'].tolist()

# Prepare containers
evidence_T = {}  # will hold per-timestamp details
total_allowed_rows = 50
per_timestamp_cap = max(1, total_allowed_rows // max(1, len(ts_list)))

# Helper to fetch original rows for an event
def fetch_original_rows_for_event(ts, event_tuple, max_rows_for_event):
    cmdb_id, source, name, breach_type = event_tuple
    rows = pd.DataFrame()
    if source == 'trace':
        df = df_trace_incident
        name_col = 'trace_name'
    elif source == 'log':
        df = df_log_incident
        name_col = 'log_name'
    elif source == 'metric':
        df = df_metric_incident
        name_col = 'kpi_name'
    else:
        return pd.DataFrame()

    if df.empty:
        return pd.DataFrame()

    # exact match first
    mask_exact = (df['cmdb_id'] == cmdb_id) & (df[name_col] == name) & (df['timestamp'] == ts)
    rows = df.loc[mask_exact, ['timestamp','cmdb_id', name_col, 'value']].rename(columns={name_col:'name'})

    if rows.shape[0] == 0:
        # fallback to +/-60 seconds window
        mask_win = (df['cmdb_id'] == cmdb_id) & (df[name_col] == name) & (df['timestamp'] >= ts-60) & (df['timestamp'] <= ts+60)
        rows = df.loc[mask_win, ['timestamp','cmdb_id', name_col, 'value']].rename(columns={name_col:'name'})

    if rows.shape[0] > 0:
        # sort by timestamp ascending and limit
        rows = rows.sort_values('timestamp').head(max_rows_for_event).reset_index(drop=True)
    return rows

# Iterate top timestamps
for idx, row in top_ts_df.iterrows():
    ts = int(row['timestamp'])
    events = row.get('events', []) if 'events' in row else []
    events = list(events)  # ensure list
    # limit events list to 50 to keep compact (though usually smaller)
    events = events[:50]

    collected_rows = []
    collected_dfs = []
    rows_taken = 0
    # per-event cap: up to 5 rows, but ensure overall per-timestamp cap respected
    for ev in events:
        if rows_taken >= per_timestamp_cap:
            break
        max_for_event = min(5, per_timestamp_cap - rows_taken)
        ev_rows = fetch_original_rows_for_event(ts, ev, max_for_event)
        if not ev_rows.empty:
            collected_dfs.append(ev_rows)
            rows_taken += ev_rows.shape[0]
    if collected_dfs:
        df_evi = pd.concat(collected_dfs, ignore_index=True, sort=False)
    else:
        df_evi = pd.DataFrame(columns=['timestamp','cmdb_id','name','value'])

    # compute aggregates based on events list (not limited to fetched rows)
    cmdbs = [e[0] for e in events]
    sources = [e[1] for e in events]
    distinct_component_count = len(set(cmdbs))
    counts_by_source = dict(Counter(sources))
    comp_counts = Counter(cmdbs)
    top_component = comp_counts.most_common(1)[0][0] if comp_counts else None

    evidence_T[ts] = {
        'events': events,
        'df': df_evi,  # DataFrame of original rows for this timestamp
        'aggregate': {
            'distinct_component_count': distinct_component_count,
            'counts_by_source': counts_by_source,
            'top_component_by_event_count': top_component
        }
    }

# Store df_evidence_T1 and df_evidence_T2 as requested (may be empty DataFrames if missing)
if len(ts_list) >= 1:
    t1 = ts_list[0]
    df_evidence_T1 = evidence_T[t1]['df']
else:
    t1 = None
    df_evidence_T1 = pd.DataFrame(columns=['timestamp','cmdb_id','name','value'])

if len(ts_list) >= 2:
    t2 = ts_list[1]
    df_evidence_T2 = evidence_T[t2]['df']
else:
    t2 = None
    df_evidence_T2 = pd.DataFrame(columns=['timestamp','cmdb_id','name','value'])

# Prepare output payload, convert DataFrames to list-of-dicts but keep compact (they are already limited)
def df_to_records_limited(df, limit=50):
    if df is None or df.empty:
        return []
    return df.head(limit).to_dict(orient='records')

payload = {
    'T1': int(t1) if t1 is not None else None,
    'T2': int(t2) if t2 is not None else None,
    'T1_events': evidence_T[t1]['events'] if t1 is not None else [],
    'T1_original_rows': df_to_records_limited(df_evidence_T1, limit=50),
    'T1_aggregate': evidence_T[t1]['aggregate'] if t1 is not None else {},
    'T2_events': evidence_T[t2]['events'] if t2 is not None else [],
    'T2_original_rows': df_to_records_limited(df_evidence_T2, limit=50),
    'T2_aggregate': evidence_T[t2]['aggregate'] if t2 is not None else {}
}

# Ensure variables are stored in kernel for follow-up
df_evidence_T1, df_evidence_T2, payload

```
Out[8]:
```
Summary of extracted evidence for the top two correlated timestamps:

T1 = 1616489280 (2021-03-23 16:48:00 UTC+8) — earlier of the two
- Distinct components: 6
- Total events: 23
- Event sources breakdown: trace (15), log (4), metric (4)
- Top component by event count: MG01
- Key events (examples): 
  - IG01: trace.to_Tomcat04.duration_mean (above)
  - MG01: many trace events (from_Tomcat03.row_count, from_Tomcat04.duration_mean, self.row_count, etc.)
  - MG02: multiple trace row_count events and a tiny below for from_dockerA1.duration_mean
  - Tomcat02/03/04: log.gc_pause_max and log.row_count (above)
  - Metric anomalies: IG01 JVM heap memory (above), MG01 JVM heap (below), MG01 CPU WIO (above), MG01 CPU idle (below)
- Original rows (sampled, up to 23 rows returned): include the exact metric/trace/log values at timestamp 1616489280 (e.g., IG01 trace.to_Tomcat04.duration_mean = 0.588, MG01 trace.self.row_count = 1574, Tomcat03 log.row_count = 510, IG01 JVM heap ≈ 8.915968e8, MG01 CPU WIO = 0.9235, etc.)

T2 = 1616489340 (2021-03-23 16:49:00 UTC+8)
- Distinct components: 6
- Total events: 16
- Event sources breakdown: trace (10), log (3), metric (3)
- Top component by event count: MG02
- Key events (examples):
  - IG01: trace.self.row_count and multiple trace.to_TomcatX.row_count (above)
  - MG01/MG02: several trace row_count events (above)
  - Tomcat02/04: log.row_count (above) and a gc_pause_max (one entry)
  - Metric anomalies: IG02 JVM heap (below), MG02 CPU sys/user (above)
- Original rows (sampled, up to 16 rows returned): include values at 1616489340 (e.g., IG01 trace.self.row_count = 2028, MG02 trace.self.row_count = 1542, Tomcat02 log.row_count = 508, MG02 CPU sys = 1.2031, etc.)

Interpretation / next steps (concise):
- Both timestamps show tightly clustered, multi-source anomalies (traces + logs + metrics) involving IG and MG tiers and Tomcat frontends.
- T1 (16:48) is dominated by MG01-related activity and large IG01→Tomcat trace volumes plus metric indications (IG JVM heap high, MG01 CPU/heap signals).
- T2 (16:49) is dominated by MG02 and IG01 trace bursts with matching Tomcat log.row_count spikes and some MG02 CPU metric rises.
- Action: prioritize inspecting IG01→Tomcat request bursts and MG01/MG02 resource behavior around 16:48–16:49 (check request traces, JVM heap details, Tomcat throughput and GC around these timestamps).

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id                                       name         value
0   1616489280      IG01            trace.to_Tomcat04.duration_mean  5.880620e-01
1   1616489280      MG01              trace.from_Tomcat03.row_count  2.630000e+02
2   1616489280      MG01          trace.from_Tomcat04.duration_mean  4.635700e-01
3   1616489280      MG01              trace.from_dockerB1.row_count  4.050000e+02
4   1616489280      MG01              trace.from_dockerB2.row_count  1.430000e+02
..         ...       ...                                        ...           ...
18  1616489280  Tomcat04                              log.row_count  4.720000e+02
19  1616489280      IG01  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.915968e+08
20  1616489280      MG01  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  2.090221e+08
21  1616489280      MG01                     OSLinux-CPU_CPU_CPUWio  9.235000e-01
22  1616489280      MG01                OSLinux-CPU_CPU_CPUidleutil  7.103460e+01

[23 rows x 4 columns],      timestamp   cmdb_id                                       name         value
0   1616489340      IG01                       trace.self.row_count  2.028000e+03
1   1616489340      IG01                trace.to_Tomcat01.row_count  2.600000e+02
2   1616489340      IG01                trace.to_Tomcat02.row_count  2.600000e+02
3   1616489340      IG01                trace.to_Tomcat04.row_count  2.520000e+02
4   1616489340      MG01                trace.to_dockerB1.row_count  3.910000e+02
5   1616489340      MG02              trace.from_Tomcat02.row_count  2.770000e+02
6   1616489340      MG02              trace.from_Tomcat04.row_count  2.720000e+02
7   1616489340      MG02                       trace.self.row_count  1.542000e+03
8   1616489340      MG02                trace.to_dockerB1.row_count  4.350000e+02
9   1616489340      MG02                trace.to_dockerB2.row_count  1.500000e+02
10  1616489340  Tomcat02                              log.row_count  5.080000e+02
11  1616489340  Tomcat04                           log.gc_pause_max  2.000000e-02
12  1616489340  Tomcat04                              log.row_count  5.010000e+02
13  1616489340      IG02  JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  5.730682e+07
14  1616489340      MG02                 OSLinux-CPU_CPU_CPUSysTime  1.203100e+00
15  1616489340      MG02                OSLinux-CPU_CPU_CPUUserTime  2.640610e+01, {'T1': 1616489280, 'T2': 1616489340, 'T1_events': [('IG01', 'trace', 'trace.to_Tomcat04.duration_mean', 'above'), ('MG01', 'trace', 'trace.from_Tomcat03.row_count', 'above'), ('MG01', 'trace', 'trace.from_Tomcat04.duration_mean', 'above'), ('MG01', 'trace', 'trace.from_dockerB1.row_count', 'above'), ('MG01', 'trace', 'trace.from_dockerB2.row_count', 'above'), ('MG01', 'trace', 'trace.self.row_count', 'above'), ('MG01', 'trace', 'trace.to_dockerB1.duration_mean', 'above'), ('MG01', 'trace', 'trace.to_dockerB2.row_count', 'above'), ('MG02', 'trace', 'trace.from_Tomcat04.row_count', 'above'), ('MG02', 'trace', 'trace.from_dockerA1.duration_mean', 'below'), ('MG02', 'trace', 'trace.from_dockerB1.row_count', 'above'), ('MG02', 'trace', 'trace.from_dockerB2.row_count', 'above'), ('MG02', 'trace', 'trace.self.row_count', 'above'), ('MG02', 'trace', 'trace.to_dockerB1.row_count', 'above'), ('MG02', 'trace', 'trace.to_dockerB2.row_count', 'above'), ('Tomcat02', 'log', 'log.gc_pause_max', 'above'), ('Tomcat03', 'log', 'log.gc_pause_max', 'above'), ('Tomcat03', 'log', 'log.row_count', 'above'), ('Tomcat04', 'log', 'log.row_count', 'above'), ('IG01', 'metric', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'above'), ('MG01', 'metric', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'below'), ('MG01', 'metric', 'OSLinux-CPU_CPU_CPUWio', 'above'), ('MG01', 'metric', 'OSLinux-CPU_CPU_CPUidleutil', 'below')], 'T1_original_rows': [{'timestamp': 1616489280, 'cmdb_id': 'IG01', 'name': 'trace.to_Tomcat04.duration_mean', 'value': 0.588062}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.from_Tomcat03.row_count', 'value': 263.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.from_Tomcat04.duration_mean', 'value': 0.46357}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.from_dockerB1.row_count', 'value': 405.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.from_dockerB2.row_count', 'value': 143.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.self.row_count', 'value': 1574.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.to_dockerB1.duration_mean', 'value': 0.576541}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'trace.to_dockerB2.row_count', 'value': 121.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.from_Tomcat04.row_count', 'value': 252.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.from_dockerA1.duration_mean', 'value': 0.006611}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.from_dockerB1.row_count', 'value': 407.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.from_dockerB2.row_count', 'value': 138.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.self.row_count', 'value': 1595.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.to_dockerB1.row_count', 'value': 376.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG02', 'name': 'trace.to_dockerB2.row_count', 'value': 134.0}, {'timestamp': 1616489280, 'cmdb_id': 'Tomcat02', 'name': 'log.gc_pause_max', 'value': 0.14}, {'timestamp': 1616489280, 'cmdb_id': 'Tomcat03', 'name': 'log.gc_pause_max', 'value': 0.13}, {'timestamp': 1616489280, 'cmdb_id': 'Tomcat03', 'name': 'log.row_count', 'value': 510.0}, {'timestamp': 1616489280, 'cmdb_id': 'Tomcat04', 'name': 'log.row_count', 'value': 472.0}, {'timestamp': 1616489280, 'cmdb_id': 'IG01', 'name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'value': 891596800.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'value': 209022064.0}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'OSLinux-CPU_CPU_CPUWio', 'value': 0.9235}, {'timestamp': 1616489280, 'cmdb_id': 'MG01', 'name': 'OSLinux-CPU_CPU_CPUidleutil', 'value': 71.0346}], 'T1_aggregate': {'distinct_component_count': 6, 'counts_by_source': {'trace': 15, 'log': 4, 'metric': 4}, 'top_component_by_event_count': 'MG01'}, 'T2_events': [('IG01', 'trace', 'trace.self.row_count', 'above'), ('IG01', 'trace', 'trace.to_Tomcat01.row_count', 'above'), ('IG01', 'trace', 'trace.to_Tomcat02.row_count', 'above'), ('IG01', 'trace', 'trace.to_Tomcat04.row_count', 'above'), ('MG01', 'trace', 'trace.to_dockerB1.row_count', 'above'), ('MG02', 'trace', 'trace.from_Tomcat02.row_count', 'above'), ('MG02', 'trace', 'trace.from_Tomcat04.row_count', 'above'), ('MG02', 'trace', 'trace.self.row_count', 'above'), ('MG02', 'trace', 'trace.to_dockerB1.row_count', 'above'), ('MG02', 'trace', 'trace.to_dockerB2.row_count', 'above'), ('Tomcat02', 'log', 'log.row_count', 'above'), ('Tomcat04', 'log', 'log.gc_pause_max', 'below'), ('Tomcat04', 'log', 'log.row_count', 'above'), ('IG02', 'metric', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'below'), ('MG02', 'metric', 'OSLinux-CPU_CPU_CPUSysTime', 'above'), ('MG02', 'metric', 'OSLinux-CPU_CPU_CPUUserTime', 'above')], 'T2_original_rows': [{'timestamp': 1616489340, 'cmdb_id': 'IG01', 'name': 'trace.self.row_count', 'value': 2028.0}, {'timestamp': 1616489340, 'cmdb_id': 'IG01', 'name': 'trace.to_Tomcat01.row_count', 'value': 260.0}, {'timestamp': 1616489340, 'cmdb_id': 'IG01', 'name': 'trace.to_Tomcat02.row_count', 'value': 260.0}, {'timestamp': 1616489340, 'cmdb_id': 'IG01', 'name': 'trace.to_Tomcat04.row_count', 'value': 252.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG01', 'name': 'trace.to_dockerB1.row_count', 'value': 391.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'trace.from_Tomcat02.row_count', 'value': 277.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'trace.from_Tomcat04.row_count', 'value': 272.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'trace.self.row_count', 'value': 1542.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'trace.to_dockerB1.row_count', 'value': 435.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'trace.to_dockerB2.row_count', 'value': 150.0}, {'timestamp': 1616489340, 'cmdb_id': 'Tomcat02', 'name': 'log.row_count', 'value': 508.0}, {'timestamp': 1616489340, 'cmdb_id': 'Tomcat04', 'name': 'log.gc_pause_max', 'value': 0.02}, {'timestamp': 1616489340, 'cmdb_id': 'Tomcat04', 'name': 'log.row_count', 'value': 501.0}, {'timestamp': 1616489340, 'cmdb_id': 'IG02', 'name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'value': 57306824.0}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'OSLinux-CPU_CPU_CPUSysTime', 'value': 1.2031}, {'timestamp': 1616489340, 'cmdb_id': 'MG02', 'name': 'OSLinux-CPU_CPU_CPUUserTime', 'value': 26.4061}], 'T2_aggregate': {'distinct_component_count': 6, 'counts_by_source': {'trace': 10, 'log': 3, 'metric': 3}, 'top_component_by_event_count': 'MG02'}})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```